In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS credit_risk_analysis_db.audit_sch;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS credit_risk_analysis_db.audit_sch.pipeline_audit
(
table_name STRING,

layer STRING,

pipeline_name STRING,

start_time TIMESTAMP,

end_time TIMESTAMP,

status STRING,

source_count BIGINT,

target_count BIGINT,

records_loaded BIGINT,

records_rejected BIGINT,

batch_id STRING,

execution_time_seconds DOUBLE,

executed_by STRING,

remarks STRING
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *
from datetime import datetime

def start_audit(table_name, layer, pipeline):

    return {
        "table_name": table_name,
        "layer": layer,
        "pipeline_name": pipeline,
        "start_time": datetime.now()
    }


def end_audit(
    audit,
    source_count,
    target_count,
    records_loaded,
    records_rejected,
    status,
    batch_id,
    remarks=""
):

    end_time = datetime.now()

    execution_time = (
        end_time - audit["start_time"]
    ).total_seconds()

    audit_df = spark.createDataFrame([(

        audit["table_name"],

        audit["layer"],

        audit["pipeline_name"],

        audit["start_time"],

        end_time,

        status,

        source_count,

        target_count,

        records_loaded,

        records_rejected,

        batch_id,

        execution_time,

        spark.sql("SELECT current_user()").first()[0],

        remarks

    )], [

        "table_name",

        "layer",

        "pipeline_name",

        "start_time",

        "end_time",

        "status",

        "source_count",

        "target_count",

        "records_loaded",

        "records_rejected",

        "batch_id",

        "execution_time_seconds",

        "executed_by",

        "remarks"

    ])

    (
        audit_df.write
        .mode("append")
        .saveAsTable("credit_risk_analysis_db.audit_sch.pipeline_audit")
    )

In [0]:
audit = start_audit(

table_name,

"Bronze",

"Bronze Load"

)

In [0]:
source_count = df.count()

target_count = bronze_df.count()

loaded = target_count

rejected = source_count - target_count

In [0]:
end_audit(

audit,

source_count,

target_count,

loaded,

rejected,

"SUCCESS",

str(uuid.uuid4()),

"Bronze Load Completed"

)

In [0]:
except Exception as e:

    end_audit(

        audit,

        0,

        0,

        0,

        0,

        "FAILED",

        str(uuid.uuid4()),

        str(e)

    )

    raise

In [0]:
audit = start_audit(

table_name,

"Silver",

"Silver Load"

)

In [0]:
display(

spark.table(

"credit_risk_analysis_db.audit_sch.pipeline_audit"

)
)